##### Wikipedia Retriever

In [1]:
from langchain_community.retrievers import WikipediaRetriever

/var/folders/vn/j3kyb3f14dngz6y0dp2yf5jh0000gn/T/ipykernel_78638/2879121110.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [2]:
#initialize the retriever
retriever = WikipediaRetriever( #args are optional
    top_k_results=2,
    lang="en"
)

In [10]:
#define the query 
query = "Data Centers environment crisis" #give shortest possible key-word heavy arguments as it doesn't perform semantic search but key word based retrieval

#fetch relevent docs from wikipedia
docs = retriever.invoke(query)

In [11]:
print(docs[0].page_content)

An AI data center is a specialized data center facility designed for the computationally intensive tasks of training and running inference for artificial intelligence (AI) and machine learning models. Unlike general-purpose data centers, they are optimized for the parallel processing demands of AI workloads, typically using hardware such as AI accelerators (e.g., GPUs, TPUs) and high-speed interconnects.
The global push to construct these specialized facilities accelerated dramatically during the AI boom of the 2020s. Memory manufacturers prioritized production of High Bandwidth Memory (HBM) essential for AI servers, which led to a global memory supply shortage amid a broader competition for advanced chips, power, and infrastructure.  70% of memory production in 2026 is going to data centers, primarily AI data centers.  Quote from article: "..., AI factories are pre-booking entire year's worth of supply, leading to severe shortages in the smartphone and PC segments. And the majority of

#### Vector Store Retriever
    This is the most used one.

In [2]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

/Users/ashutosh/projects/LangChain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
    Document(page_content="LangChain makes it easy to work with LLMs"),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [4]:
#initialize the model
embd = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L6-v2')

#create chroma vector store in memo
vector_store = Chroma.from_documents(
    documents = documents,
    embedding=embd,
    collection_name ="my_collec"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9108.44it/s]


In [21]:
#convert vector store into a retriever
retriever = vector_store.as_retriever(search_kwargs = {"k":2})

In [23]:
query = "What is chroma used for ? "

results = retriever.invoke(query) #note invoke() -> a runnable

In [24]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


One Common Doubt: Why not use Vector stores directly to fetxh results?
1. Chain integration aka Runnable
2. (More solid reason) -> retrievers allow many variety of advanced strategies of retrieving relevent documents instead of just similarity search provided by vector stores

In [5]:
from langchain_community.vectorstores import FAISS

embd = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L6-v2')

vector_store = FAISS.from_documents(
    documents = documents,
    embedding=embd
)

retriever = vector_store.as_retriever(
    search_type = "mmr",  #this enables MMR
    search_kwargs = {"k" : 3, "lambda" : 0.5}
)

res = retriever.invoke("what is langchain?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15573.10it/s]


In [36]:
for i, doc in enumerate(res):
    print(f"---Result-{i+1}---")
    print(doc.page_content)
    

---Result-1---
LangChain supports Chroma, FAISS, Pinecone, and more.
---Result-2---
LangChain helps developers build LLM applications easily.
---Result-3---
Embeddings convert text into high-dimensional vectors.


In [6]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever #depricated, not used now, Langgraph multiple pipelines do this task
from langchain_groq import ChatGroq

# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [42]:
# Create FAISS vector store
vectorstoree = FAISS.from_documents(documents=all_docs, embedding=embd)

In [ ]:
# Create retrievers -> first based on similarity search
similarity_retriever = vectorstoree.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstoree.as_retriever(search_kwargs={"k": 5}), #5 outputs per each sample prompt, which is later de-duplicated
    llm=ChatGroq(model="openai/gpt-oss-20b")
)

In [18]:
query = "How to improve energy levels and maintain balance?"
simple = similarity_retriever.invoke(query)
multi = multiquery_retriever.invoke(query)

In [13]:
print("Simple Similarity Retriever Results")
for i, doc in enumerate(simple):
    print(f"---result-{i+1}---")
    print(doc.page_content)

Simple Similarity Retriever Results
---result-1---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
---result-2---
The solar energy system in modern homes helps balance electricity demand.
---result-3---
Consuming leafy greens and fruits helps detox the body and improve longevity.
---result-4---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
---result-5---
Photosynthesis enables plants to produce energy by converting sunlight.


In [19]:
print("Multi Query Similarity Retriever Results")
for i, doc in enumerate(multi):
    print(f"---result-{i+1}---")
    print(doc.page_content)

Multi Query Similarity Retriever Results
---result-1---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
---result-2---
The solar energy system in modern homes helps balance electricity demand.
---result-3---
Consuming leafy greens and fruits helps detox the body and improve longevity.
---result-4---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
---result-5---
Regular walking boosts heart health and can reduce symptoms of depression.
---result-6---
Deep sleep is crucial for cellular repair and emotional regulation.
---result-7---
Photosynthesis enables plants to produce energy by converting sunlight.


##### Contextual Compression Retriever
The standard wrapper has been depricated hence using ChatPromptTemplate to prepare a template and ask ChatGroq LLM to trim the docs for us and wrapping everything in a runnableLambda to include in LCEL

In [43]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS

embd = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L6-v2')
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7726.39it/s]


In [47]:
vector_store = FAISS.from_documents(
    documents = docs,
    embedding=embd
)

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# 2. Define the Compression Prompt
compression_prompt = ChatPromptTemplate.from_template(
    "You are an expert information extractor.\n"
    "Given the following document excerpt, extract ONLY the exact sentences or facts "
    "that are directly relevant to answering the user's query.\n"
    "Do not add commentary, summaries, or introductory text. If the document is completely irrelevant, return nothing.\n\n"
    "Query: {query}\n"
    "Document Excerpt:\n{doc_content}\n\n"
    "Extracted Relevant Text:"
)

# Create a mini-chain just for compressing a single chunk of text
compression_chain = compression_prompt | llm | StrOutputParser()

# 3. Write the Custom Compressor Function
def custom_compressor(inputs: dict) -> list[Document]:
    query = inputs["query"]
    
    # Step A: Fetch raw documents from Chroma
    raw_docs = base_retriever.invoke(query)
    compressed_docs = []
    
    # Step B: Loop through and compress each document text using the LLM
    for doc in raw_docs:
        extracted_content = compression_chain.invoke({
            "query": query,
            "doc_content": doc.page_content
        })
        
        # Clean up whitespace and skip if the LLM deemed it irrelevant
        cleaned_content = extracted_content.strip()
        if cleaned_content and "not relevant" not in cleaned_content.lower():
            # Retain original metadata so your sources remain intact!
            compressed_docs.append(
                Document(page_content=cleaned_content, metadata=doc.metadata)
            )
            
    return compressed_docs

# 4. Wrap it in a RunnableLambda to make it LCEL-compatible
compressor_runnable = RunnableLambda(custom_compressor)

# 5. Build your Final RAG Pipeline
# This takes the query, passes it to your compressor, and outputs compressed docs
rag_retrieval_pipe = {"query": RunnablePassthrough()} | compressor_runnable


##### NOTE: -> for "{"query": RunnablePassthrough()}"

Running compressor_runnable.invoke({"query": "your query"}) works perfectly and gives you the exact same result.The reason developers use the extra {"query": RunnablePassthrough()} wrapper is not for the compressor itself, but for chaining it with other components later on.

In [52]:
res = rag_retrieval_pipe.invoke("what is photosynthesis?")

In [54]:
for doc in res:
    print("-----res------")
    print(doc.page_content,"\n")  #only returns the relevant part from the big chunks of every single doc fetched by standard retriever

-----res------
Photosynthesis is the process by which green plants convert sunlight into energy. 

-----res------
The chlorophyll in plant cells captures sunlight during photosynthesis. 

-----res------
Photosynthesis does not occur in animal cells. 

